# 01 · Data Pipeline (Phase 1)

**What this notebook does:** coordinates Phase 1 of `project.md`. It downloads the SEC insider filings, applies the Section 5.2 filters in order, fetches prices, and saves the outputs. All logic lives in `src/`; this notebook only calls functions and shows results.

**Inputs:** SEC quarterly zips (cached in `data/raw/sec/`), yfinance prices (cached in `data/raw/prices/`).

**Outputs:**
| File | Contents |
|---|---|
| `data/processed/purchases.parquet` | one row per insider purchase filing that passed filters 1–7 |
| `results/phase1_filters/row_counts.csv` | row count after every filter step |
| `results/phase1_filters/price_match_drops.csv` | tickers dropped at filter 7, with reason |
| `results/phase1_filters/price_match_by_year.csv` | share of purchases lost to missing prices, by filing year |
| `results/phase1_filters/spot_check_sample.csv` | 10 random purchases with EDGAR links for manual checking |
| `results/phase1_filters/openinsider_check_sample.csv` | 10 recent purchases (last 3 months) to compare against OpenInsider |

The first full run downloads ~82 SEC zips (about 1 GB) and prices for several thousand tickers, so it takes a while. Re-runs use the cache.

## Step 0 · Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from utils import PROCESSED_DIR, save_results_csv
from load_filings import list_quarters, load_all_purchases
from load_prices import (
    MARKET_BENCHMARK,
    apply_price_filters,
    attach_close_on_filing_date,
    clean_ticker,
    download_prices,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

## Step 1 · Choose quarters
Every available quarter, 2006q1 through 2026q2 (confirmed on the SEC page on 26 Sep 2026).

In [ ]:
quarters_to_run = list_quarters()
print(f"{len(quarters_to_run)} quarters: {quarters_to_run[0]} ... {quarters_to_run[-1]}")

## Step 2 · Download filings and apply filters 1–5
Filters, in order (project.md Section 5.2, plus decisions logged in project.md Section 12):
1. Form 4 only; 4/A amendments dropped (the readme gives no de-duplication rule)
2. Non-derivative transactions whose security title contains "common" or "ordinary"
3. Transaction code P, acquired (A)
   - then the lots of each filing are summed into one row, and the first-listed owner is attached
4. Not flagged as a Rule 10b5-1 plan (flag exists from 2023; earlier = unknown, kept)
5. Total purchase value ≥ $10,000

In [ ]:
purchases, row_count_table = load_all_purchases(quarters_to_run)
row_count_table

## Step 3 · Download prices (every purchase ticker + SPY)
Tickers are cleaned from the as-filed symbol (see `clean_ticker`). Prices are cached per ticker; tickers Yahoo has no data for are remembered so they are not requested again.

In [ ]:
purchase_tickers = purchases["ticker_as_filed"].apply(clean_ticker).dropna().unique().tolist()
download_prices(purchase_tickers + [MARKET_BENCHMARK])

## Step 4 · Apply filters 6–7
6. As-traded close on the filing date ≥ $2. Yahoo's close is split-adjusted, so the actual traded price is rebuilt from the split history first.
7. Keep only purchases matched to price data (last close on or up to 5 days before the filing date).

   7b. Keep only purchases whose filed price is within 50% of Yahoo's as-traded close (catches filing typos and reused tickers).

In [ ]:
row_count_log = row_count_table.to_dict("records")
purchases_with_prices, price_match_drops = apply_price_filters(purchases, row_count_log)
row_count_table = pd.DataFrame(row_count_log)
row_count_table

## Step 5 · Survivor-bias check: price match rate by filing year
project.md Section 5.3: if more than 20% of events are lost to missing prices, flag it prominently.

In [ ]:
price_status = attach_close_on_filing_date(
    purchases.assign(ticker=purchases["ticker_as_filed"].apply(clean_ticker))
)
price_status["filing_year"] = price_status["filing_date"].dt.year
price_status["lost_to_missing_price"] = price_status["price_match_status"] != "matched"

price_match_by_year = (
    price_status.groupby("filing_year")
    .agg(
        purchases_after_filter_5=("lost_to_missing_price", "size"),
        lost_to_missing_price=("lost_to_missing_price", "sum"),
    )
    .reset_index()
)
price_match_by_year["share_lost"] = (
    price_match_by_year["lost_to_missing_price"] / price_match_by_year["purchases_after_filter_5"]
)

overall_share_lost = price_status["lost_to_missing_price"].mean()
print(f"Overall share of purchases lost to missing prices: {overall_share_lost:.1%}")
if overall_share_lost > 0.20:
    print("FLAG: more than 20% lost to missing prices (project.md 5.3). Survivor-bias risk.")
price_match_by_year

## Step 6 · Save outputs

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
purchases_with_prices.to_parquet(PROCESSED_DIR / "purchases.parquet", index=False)

print(save_results_csv(row_count_table, "phase1_filters", "row_counts.csv"))
print(save_results_csv(price_match_drops, "phase1_filters", "price_match_drops.csv"))
print(save_results_csv(price_match_by_year, "phase1_filters", "price_match_by_year.csv"))
print(f"{len(purchases_with_prices):,} purchases saved")

## Step 7 · Spot-check sample for Hunter
10 random purchases (fixed seed so the sample is reproducible). Open each `edgar_link` and check the owner, date, shares and price against the filing.

In [ ]:
spot_check_sample = purchases_with_prices.sample(10, random_state=2026)
spot_check_sample = spot_check_sample.assign(
    edgar_link="https://www.sec.gov/Archives/edgar/data/"
    + spot_check_sample["issuer_cik"].str.lstrip("0")
    + "/"
    + spot_check_sample["accession_number"].str.replace("-", "")
    + "/"
    + spot_check_sample["accession_number"]
    + "-index.htm"
)
spot_check_columns = [
    "ticker", "issuer_name", "owner_name", "filing_date", "first_transaction_date",
    "number_of_lots", "shares_purchased", "average_price", "purchase_value_usd", "edgar_link",
]
spot_check_sample = spot_check_sample[spot_check_columns]
print(save_results_csv(spot_check_sample, "phase1_filters", "spot_check_sample.csv"))
spot_check_sample

## Step 8 · OpenInsider cross-check sample for Hunter
project.md Section 7: 10 purchases filed in the last 3 months of the data. For each, open `openinsider_link` and compare ticker, insider title, share count and dollar value. Manual check only: no scraping (Section 9).

In [ ]:
latest_filing_date = purchases_with_prices["filing_date"].max()
three_months_before_latest = latest_filing_date - pd.DateOffset(months=3)
recent_purchases = purchases_with_prices[purchases_with_prices["filing_date"] > three_months_before_latest]
print(f"{len(recent_purchases):,} purchases filed {three_months_before_latest.date()} to {latest_filing_date.date()}")

openinsider_check_sample = recent_purchases.sample(10, random_state=2026)
openinsider_check_sample = openinsider_check_sample.assign(
    openinsider_link="http://openinsider.com/screener?s=" + openinsider_check_sample["ticker"]
)
openinsider_check_columns = [
    "ticker", "owner_name", "owner_relationship", "owner_title", "filing_date",
    "first_transaction_date", "shares_purchased", "average_price", "purchase_value_usd", "openinsider_link",
]
openinsider_check_sample = openinsider_check_sample[openinsider_check_columns]
print(save_results_csv(openinsider_check_sample, "phase1_filters", "openinsider_check_sample.csv"))
openinsider_check_sample